# Переменные и библиотеки

In [24]:
import cv2
import numpy as np

OUT_WIDTH = 1000
OUT_HEIGHT = 700
CAMERA_INDEX = 0

# Aruco
aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
parameters = cv2.aruco.DetectorParameters()
detector = cv2.aruco.ArucoDetector(aruco_dict, parameters)

# Функции

In [ ]:
def get_marker_ordered_crn(pts_4x2, axis_x, axis_y):
    """Упорядочивает 4 угла одного ArUco-маркера относительно осей поля"""
    center = np.mean(pts_4x2, axis=0)
    tl = tr = br = bl = center
    for pt in pts_4x2:
        dx = float(np.dot(pt - center, axis_x))
        dy = float(np.dot(pt - center, axis_y))
        if dx < 0 and dy < 0:
            tl = pt
        elif dx >= 0 and dy < 0:
            tr = pt
        elif dx >= 0 and dy >= 0:
            br = pt
        else:
            bl = pt
    return (tuple(tl), tuple(tr), tuple(br), tuple(bl))

def build_exact_homography(crn_seq):
    """Строит точную перспективную гомографию по всем 12 углам 3 маркеров"""
    it = iter(crn_seq)
    m0 = next(it)
    m1 = next(it)
    m2 = next(it)

    c0 = np.mean(m0, axis=0)
    c1 = np.mean(m1, axis=0)
    c2 = np.mean(m2, axis=0)

    d01 = float(np.linalg.norm(c0 - c1))
    d12 = float(np.linalg.norm(c1 - c2))
    d02 = float(np.linalg.norm(c0 - c2))

    if d01 < 10 or d12 < 10 or d02 < 10:
        return None

    cos0 = abs(float(np.dot(c1 - c0, c2 - c0))) / (d01 * d02)
    cos1 = abs(float(np.dot(c0 - c1, c2 - c1))) / (d01 * d12)
    cos2 = abs(float(np.dot(c0 - c2, c1 - c2))) / (d02 * d12)

    if cos0 <= cos1 and cos0 <= cos2:
        m_br, c_br = m0, c0
        ma, ca, mb, cb = m1, c1, m2, c2
    elif cos1 <= cos0 and cos1 <= cos2:
        m_br, c_br = m1, c1
        ma, ca, mb, cb = m0, c0, m2, c2
    else:
        m_br, c_br = m2, c2
        ma, ca, mb, cb = m0, c0, m1, c1

    v1 = ca - c_br
    v2 = cb - c_br
    cross = float(np.cross(v1, v2))

    if cross < 0:
        m_tr, c_tr = ma, ca
        m_bl, c_bl = mb, cb
    else:
        m_tr, c_tr = mb, cb
        m_bl, c_bl = ma, ca

    axis_x = c_br - c_bl
    axis_y = c_br - c_tr

    src_bl = get_marker_ordered_crn(m_bl, axis_x, axis_y)
    src_br = get_marker_ordered_crn(m_br, axis_x, axis_y)
    src_tr = get_marker_ordered_crn(m_tr, axis_x, axis_y)

    src_pts = np.array(src_bl + src_br + src_tr, dtype = "float32")

    dst_pts = np.array((
        (54.0, 613.0), (119.0, 613.0), (119.0, 678.0), (54.0, 678.0),
        (805.0, 613.0), (870.0, 613.0), (870.0, 678.0), (805.0, 678.0),
        (897.0, 51.0), (962.0, 51.0), (962.0, 116.0), (897.0, 116.0)
    ), dtype = "float32")

    H, _ = cv2.findHomography(src_pts, dst_pts)
    return H

def analyze_cell(hsv_roi):
    """Определяет наличие шашки (1 или 0) и ее цвет строго в центре ячейки."""
    h, w, _ = hsv_roi.shape
    if h == 0 or w == 0:
        return 0, None, None

    center_crop = hsv_roi[int(h * 0.35):int(h * 0.65), int(w * 0.35):int(w * 0.65)]
    mean_hsv = np.mean(center_crop, axis=(0, 1))
    
    hue = float(mean_hsv.item(0))
    sat = float(mean_hsv.item(1))
    val = float(mean_hsv.item(2))

    if sat < 45:
        if val < 85:
            return 1, "Black", (50, 50, 50)
        return 0, None, None

    if (0 <= hue <= 10) or (170 <= hue <= 179):
        return 1, "Red", (0, 0, 255)
    elif 11 <= hue <= 22:
        return 1, "Orange", (0, 165, 255)
    elif 23 <= hue <= 38:
        return 1, "Yellow", (0, 255, 255)
    elif 39 <= hue <= 85:
        return 1, "Green", (0, 255, 0)
    elif 86 <= hue <= 130:
        return 1, "Blue", (255, 0, 0)
    elif 131 <= hue <= 169:
        return 1, "Purple", (255, 0, 255)
    
    return 1, "Color", (200, 200, 200)


# Main

In [ ]:
def main(): 
    cap = cv2.VideoCapture(CAMERA_INDEX)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

    if not cap.isOpened():
        print(f"Camera {CAMERA_INDEX} ERROR")
        return

    print(f"Window opened ({CAMERA_INDEX})")

    m_left = 0.014
    m_right = 0.957
    m_top = 0.162
    m_bottom = 0.942

    ls_val_mtx = None

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        crn, ids, _ = detector.detectMarkers(frame)

        if ids is not None and len(ids) >= 3:
            uniq_marks = {}
            flat_ids = ids.flatten()
            for idx in range(len(flat_ids)):
                m_id = int(flat_ids.item(idx))
                if m_id not in uniq_marks and len(uniq_marks) < 3:
                    uniq_marks[m_id] = crn[idx].reshape(4, 2)

            if len(uniq_marks) == 3:
                H = build_exact_homography(uniq_marks.values())
                if H is not None:
                    ls_val_mtx = H

        if ls_val_mtx is not None:
            warped = cv2.warpPerspective(frame, ls_val_mtx, (OUT_WIDTH, OUT_HEIGHT))
            # Переворот одновременно по вертикали и горизонтали (-1)
            warped = cv2.flip(warped, -1)
            
            hsv_warped = cv2.cvtColor(warped, cv2.COLOR_BGR2HSV)
            display_img = warped.copy()

            x_start = int(OUT_WIDTH * m_left)
            x_end = int(OUT_WIDTH * m_right)
            y_start = int(OUT_HEIGHT * m_top)
            y_end = int(OUT_HEIGHT * m_bottom)

            cell_w = (x_end - x_start) / 10.0
            cell_h = (y_end - y_start) / 6.0

            binary_grid = np.zeros((6, 10), dtype = int)

            for r in range(6):
                for c in range(10):
                    x1 = int(x_start + c * cell_w)
                    y1 = int(y_start + r * cell_h)
                    x2 = int(x1 + cell_w)
                    y2 = int(y1 + cell_h)

                    cell_hsv = hsv_warped[y1:y2, x1:x2]
                    chck, clr_nm, bgr_color = analyze_cell(cell_hsv)

                    binary_grid[r, c] = chck
                    cv2.rectangle(display_img, (x1, y1), (x2, y2), (0, 0, 255), 1)

                    if chck == 1:
                        center_x = int(x1 + cell_w / 2)
                        center_y = int(y1 + cell_h / 2)
                        cv2.circle(display_img, (center_x, center_y), 16, bgr_color, -1)
                        cv2.circle(display_img, (center_x, center_y), 16, (0, 0, 0), 2)
                        cv2.putText(display_img, clr_nm, (x1 + 8, y2 - 8),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2)

            total_count = int(np.sum(binary_grid))
            cv2.putText(display_img, f"Checkers count: {total_count}", (20, 35),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 150, 0), 2)

            cv2.imshow("Camera", display_img)
        else:
            if ids is not None:
                cv2.aruco.drawDetectedMarkers(frame, crn, ids)
            cv2.imshow("Camera", frame)

        key = cv2.waitKey(1) & 0xFF
        step = 0.002
        if key == ord('q'):
            break
        elif key == ord('w'):
            m_top = max(0.0, m_top - step)
        elif key == ord('s'):
            m_top = min(0.5, m_top + step)
        elif key == ord('i'):
            m_bottom = max(0.5, m_bottom - step)
        elif key == ord('k'):
            m_bottom = min(1.0, m_bottom + step)
        elif key == ord('a'):
            m_left = max(0.0, m_left - step)
        elif key == ord('d'):
            m_left = min(0.5, m_left + step)
        elif key == ord('j'):
            m_right = max(0.5, m_right - step)
        elif key == ord('l'):
            m_right = min(1.0, m_right + step)

    cap.release()
    cv2.destroyAllWindows()
    print("Window closed")


main()

Window opened (0)


C:\Users\Nikita\AppData\Local\Temp\ipykernel_11060\2667929385.py:52: DeprecationWarning: Arrays of 2-dimensional vectors are deprecated. Use arrays of 3-dimensional vectors instead. (deprecated in NumPy 2.0)
  cross = float(np.cross(v1, v2))


Window closed
